# Day 3 — Functions, hands on

This notebook is my playground for today's concepts: `def`/call, defaults, `*args`, `**kwargs`, return vs print, multiple return values, docstrings + `help()`, scope, lambda, and `chr()`/`ord()` (the tool behind alphabet patterns). Each code cell has a note above it saying **what to observe** and my **predicted output** — I run the cell only after predicting.

## 1. `def` and calling

**Observe:** the `def` block alone produces nothing. Output appears only when the function is *called*. Two calls = the recipe runs twice.

**Predicted output:**
```
Boil water
Add tea, milk, sugar
Chai ready!
Boil water
Add tea, milk, sugar
Chai ready!
```

In [ ]:
def make_chai():
    print("Boil water")
    print("Add tea, milk, sugar")
    print("Chai ready!")

make_chai()   # the () actually runs it
make_chai()   # written once, used again

## 2. Parameters, arguments, defaults

**Observe:** first call uses the default `sugar_spoons=2`. Second call overrides it. Third call uses *keyword arguments*, so the order I pass them stops mattering.

**Predicted output:**
```
3 cups, 2 spoons sugar
3 cups, 1 spoons sugar
2 cups, 4 spoons sugar
```

In [ ]:
def chai_order(cups, sugar_spoons=2):        # cups, sugar_spoons = parameters
    print(cups, "cups,", sugar_spoons, "spoons sugar")

chai_order(3)                                # 3 = argument; default sugar used
chai_order(3, 1)                             # default overridden by position
chai_order(sugar_spoons=4, cups=2)           # keyword args: order doesn't matter

## 3. The mutable-default trap (advanced)

**Observe:** the second call should give a fresh box with only sabzi — but roti from call 1 is still inside! The default `[]` was created **once** at `def` time and is shared by every call. Like a tiffin box that never got washed.

**Predicted output:**
```
['roti']
['roti', 'sabzi']
```

Fix: default to `None`, create the list inside the function.

In [ ]:
def add_item(item, box=[]):      # DANGER: mutable default
    box.append(item)
    return box

print(add_item("roti"))
print(add_item("sabzi"))          # surprise — same shared list!

def add_item_safe(item, box=None):
    if box is None:
        box = []                  # genuinely new list each call
    box.append(item)
    return box

## 4. `*args` — the tiffin box

**Observe:** the same function accepts 1 item, then 3 items. Inside, `items` is a **tuple**. Note the printed brackets and the trailing comma in a 1-item tuple.

**Predicted output:**
```
Packed 1 items: ('roti',)
Packed 3 items: ('roti', 'sabzi', 'achar')
```

In [ ]:
def pack_tiffin(*items):          # * collects any number of positional args
    print("Packed", len(items), "items:", items)

pack_tiffin("roti")
pack_tiffin("roti", "sabzi", "achar")

## 5. `**kwargs` — the labelled masala dabba

**Observe:** every argument arrives as `name=value`, and inside the function `spices` is a **dictionary**. Looping with `.items()` gives each name and value pair.

**Predicted output:**
```
haldi -> 1 tsp
jeera -> 2 tsp
mirchi -> half tsp
```

In [ ]:
def masala_dabba(**spices):       # ** collects any number of name=value args
    for name, qty in spices.items():
        print(name, "->", qty)

masala_dabba(haldi="1 tsp", jeera="2 tsp", mirchi="half tsp")

## 6. `return` vs `print` — the big one

**Observe carefully.** `add_print` *shows* 5 but hands back nothing, so `r1` is `None`. `add_return` shows nothing but `r2` really holds 5, so `r2 + 10` works. Trying `r1 + 10` would crash with `TypeError` (can't add `None` and a number) — that line is commented so the cell runs.

**Predicted output:**
```
5
r1 is: None
r2 is: 5
r2 + 10 = 15
```

In [ ]:
def add_print(a, b):
    print(a + b)          # shows the value, returns nothing

def add_return(a, b):
    return a + b          # hands the value back to the caller

r1 = add_print(2, 3)      # the 5 on screen comes from print, not from r1
print("r1 is:", r1)       # None — every function without return gives None

r2 = add_return(2, 3)
print("r2 is:", r2)
print("r2 + 10 =", r2 + 10)

# r1 + 10                 # uncomment to see the classic bug: TypeError with NoneType

## 7. Returning multiple values — tuple unpacking

A function can hand back several answers at once — a parcel with two items packed together. Secretly it's still **one** return value: a tuple. The comma on the `return` line packs; the commas on the left of `=` unpack.

**Observe:** printing the raw result shows a tuple `(1, 9)`; unpacking splits it into `lo` and `hi`. The commented line shows the counts-must-match rule.

**Predicted output:**
```
(1, 9)
lo: 1 hi: 9
```

In [ ]:
def min_max(nums):
    return min(nums), max(nums)    # the comma packs both into ONE tuple

print(min_max([4, 9, 1, 7]))       # the whole parcel

lo, hi = min_max([4, 9, 1, 7])     # tuple unpacking: parcel opened into two names
print("lo:", lo, "hi:", hi)

# lo, hi, extra = min_max([4, 9, 1, 7])   # uncomment -> ValueError: counts must match

## 8. Docstrings and `help()` — the label on the jar

A **docstring** is a triple-quoted string as the *first line* of a function body — the pickle-jar label saying what's inside. Python stores it, and `help(function_name)` prints it back. Bonus peek: **type hints** (`cups: int`, `-> int`) are optional notes for humans and editors; Python does not enforce them.

**Observe:** `help(chai_bill)` shows the signature and my one-line docstring — same mechanism as `help(print)`.

**Predicted output:**
```
Help on function chai_bill in module __main__:

chai_bill(cups, price_per_cup=10)
    Return the total bill for the given cups of chai.

30
```

In [ ]:
def chai_bill(cups, price_per_cup=10):
    """Return the total bill for the given cups of chai."""
    return cups * price_per_cup

help(chai_bill)                    # prints the signature + the docstring

# Modern-practice peek: type hints — notes for humans and tools, NOT enforced
def chai_bill_hinted(cups: int, price_per_cup: int = 10) -> int:
    """Same function, now with type hints."""
    return cups * price_per_cup

print(chai_bill_hinted(3))

## 9. Local vs global scope — the surprise

**Observe:** reading a global inside a function is fine. But `secret_masala` created *inside* `kitchen()` dies when the function ends — the last line (commented) would raise `NameError`. And re-assigning a global needs the `global` keyword, otherwise Python makes a brand-new local instead. (Advanced cousin for later: `nonlocal` does the same job for nested functions.)

**Predicted output:**
```
Board says: Sharma Chai Corner
Inside kitchen: kasuri methi
Cups sold: 1
```

In [ ]:
shop_name = "Sharma Chai Corner"   # global — whole file can read it
count = 0

def board():
    print("Board says:", shop_name)    # reading a global: allowed

def kitchen():
    secret_masala = "kasuri methi"     # local — exists only in here
    print("Inside kitchen:", secret_masala)

def sell():
    global count                       # without this line: UnboundLocalError
    count = count + 1

board()
kitchen()
sell()
print("Cups sold:", count)

# print(secret_masala)   # uncomment -> NameError: what happens in the kitchen stays there

## 10. Lambda — the one-line throwaway function (advanced)

Shape: `lambda inputs: expression` — no name, no `return`; the expression's value *is* the return. Like borrowing a pen just to sign, not to keep. It shines when passed into another function as a tiny rule: `key=` for sorting, or `map` for transforming.

**Observe:** the sort orders students by marks (`s[1]`), not by name. Anything bigger than one expression deserves a proper `def`.

**Predicted output:**
```
25
[('Aman', 74), ('Rahul', 82), ('Priya', 95)]
[2, 4, 6]
```

In [ ]:
square = lambda x: x * x           # same as: def square(x): return x * x
print(square(5))

students = [("Rahul", 82), ("Priya", 95), ("Aman", 74)]
students.sort(key=lambda s: s[1])  # the lambda answers: "compare by marks"
print(students)

print(list(map(lambda x: x * 2, [1, 2, 3])))   # apply a tiny rule to each item

## 11. `chr()` and `ord()` — the tool behind alphabet patterns

Computers store characters as numbers (**ASCII** codes). `'A'` is 65, `'B'` is 66 ... `'Z'` is 90. `chr(code)` turns a number into its character; `ord(char)` is the reverse.

**Observe:** counting 65, 66, 67 and converting gives A, B, C. This is exactly the trick patterns 14–15 need — a number counter wearing an alphabet costume.

**Predicted output:**
```
A B Z
65 90
ABCDE
```

In [ ]:
print(chr(65), chr(66), chr(90))     # number -> character
print(ord("A"), ord("Z"))            # character -> number (the reverse)

code = ord("A")                      # no need to memorise 65
for _ in range(5):
    print(chr(code), end="")
    code += 1                        # 65->66->67 means A->B->C
print()

---

# Practice targets — patterns 9–15 (shapes + hints only, I solve them in main.py)

All shapes are for `n = 5`. Reminders: `print(x, end="")` stays on the line; bare `print()` ends the row.

## Pattern 9 — Diamond

```
    *
   ***
  *****
 *******
*********
*********
 *******
  *****
   ***
    *
```

**Hint:** pyramid + inverted pyramid stacked — two outer loops back to back; the widest row appears twice.

## Pattern 10 — Half diamond / hourglass

```
*
**
***
****
*****
****
***
**
*
```

**Hint:** star counts climb 1→n then fall n−1→1; either two loops, or one loop of ~2n rows with an `if` picking the phase.

## Pattern 11 — Binary 0-1 triangle

```
1
0 1
1 0 1
0 1 0 1
1 0 1 0 1
```

**Hint:** even rows start with 1, odd rows with 0; inside the row, `1 - num` flips 1↔0 with no `if` needed.

## Pattern 12 — Number palindrome pyramid

```
1        1
12      21
123    321
1234  4321
1234554321
```

**Hint:** each row = count up + shrinking middle gap of spaces + count back down; the gap shrinks by 2 per row so every row is `2n` wide.

## Pattern 13 — Increasing-number triangle

```
1
2 3
4 5 6
7 8 9 10
11 12 13 14 15
```

**Hint:** one counter created **before** the outer loop and never reset — bank-token style, each row continues where the last stopped.

## Pattern 14 — Alphabet triangle

```
A
AB
ABC
ABCD
ABCDE
```

**Hint:** reset the code to `ord('A')` at the start of every row, print `chr(code)` and step +1, `i+1` letters in row `i`.

## Pattern 15 — Reverse alphabet triangle

```
ABCDE
ABCD
ABC
AB
A
```

**Hint:** same machinery as Pattern 14, but row `i` prints `n - i` letters — only the row length changes.

---

# Practice targets — prime, GCD, LCM (hints only)

## Prime check
- Exactly two divisors: 1 and itself. Handle `n < 2` first — 0 and 1 are not prime.
- Divisors come in pairs multiplying to n (for 36: 2×18, 3×12, 4×9); the smaller partner is always ≤ √n. So loop only `while i * i <= n`.
- The moment a remainder is 0, return "not prime" and exit — remember, `return` stops the function instantly.
- Test set: 1 (no), 2 (yes), 9 (no — catches a wrong boundary), 25 (no), 29 (yes).

## GCD — Euclid's algorithm
- Identity: `gcd(a, b) == gcd(b, a % b)`. Keep replacing until the second number is 0; the first is the answer.
- Loop shape in words: while b is not zero, swap the pair to (b, a % b); then return a.
- Hand trace of (48, 18) before coding: (48, 18) → (18, 12) → (12, 6) → (6, 0) → **6**.
- Feels like making change with smaller and smaller notes until nothing is left over.
- Edge check: (18, 48) — the first remainder step swaps them automatically, no special case.

## LCM
- No loop. Identity: `a * b == gcd(a, b) * lcm(a, b)`, so **lcm = (a × b) ÷ gcd(a, b)** with integer division `//`.
- Reuse the gcd function — possible *only* because gcd **returns** its answer instead of printing it. Today's whole lesson in one line.
- Test: (4, 6) → 12; (7, 5) → 35 (co-prime: gcd 1, lcm = product).